# Per-Cell Version History & AI Edit Review (`window.craftHistory`)

> **Feature 3.3:** Snapshots cell input text in memory + `sessionStorage` on every local save (`/set_contents_`) and before `/add_runq_`, surfaces an inline **Diff / Accept / Revert** bar on cells marked `data-diff="changed"` or `"deleted"`, and provides a **Cell history** modal from the cell `…` action menu.

## Templates (`#| export_ft`)

In [ ]:
#| export_ft CRAFT.js
#| eval: false
TEMPLATES = {
    'hist_modal': craft_overlay(
        craft_card(
            Div(
                Strong('Cell History (Session)'),
                Span('', cls='craft-ctx-prev-total', data_ref='cellLabel'),
                Button('×', type='button', cls='craft-modal-x', aria_label='Close', data_ref='close'),
                cls='craft-modal-head',
            ),
            Div(cls='craft-hist-list', data_ref='list'),
            Div(
                craft_btn('Close', variant='primary', size='xs', data_ref='done'),
                cls='craft-modal-foot',
            ),
            cls='craft-hist-card',
        ),
        id='craft-history-modal', role='dialog', aria_modal='true', aria_label='Cell Version History',
    ),
}

## Cell History & AI Edit Review Runtime (`#| export_js_iife`)

In [ ]:
#| export_js_iife CRAFT.js craftHistory
#| eval: false
window.craftHistory?.disable?.();
if (!document.getElementById('dialog-container')) return;

const CORE = window.__craftCMCore;
const {el, qsa, on, store, mountModal, htmxPost, onDomChange, notify, notifyErr, kid, normId, getCellEl, allCells, primaryCell, cellMtype} = CORE;
const ac = new AbortController(), sig = {signal: ac.signal};

const HIST_KEY = () => `craft-cell-hist:${kid() || location.pathname}`;
const MAX_PER_CELL = 25;

function readCellText(cell) {
  if (!cell) return null;
  const id = normId(cell.id);
  const inp = cell.querySelector(`textarea[name="content"], input[name="content"], textarea#_${id}_content`);
  if (inp && typeof inp.value === 'string') return inp.value;
  const mtype = cellMtype(cell);
  if (mtype === 'code' || mtype === 'raw') {
    const pre = cell.querySelector(':scope > .card-in .msg-content pre code, :scope > .card-in .msg-content pre');
    if (pre) return pre.textContent || '';
  }
  return null;
}

const historyMap = new Map(Object.entries(store.json(HIST_KEY(), {}, sessionStorage) || {}));
const persistHistory = () => {
  for (const limit of [MAX_PER_CELL, 12, 6, 3, 1]) {
    const obj = {};
    for (const [k, v] of historyMap) obj[k] = v.slice(-limit);
    try {
      sessionStorage.setItem(HIST_KEY(), JSON.stringify(obj));
      return;
    } catch {}
  }
};

function recordSnapshot(id, text, source = 'save') {
  const k = normId(id);
  if (!k || typeof text !== 'string') return;
  const list = historyMap.get(k) || [];
  if (list.length && list.at(-1).text === text) return;
  list.push({ts: Date.now(), source, text});
  if (list.length > MAX_PER_CELL) list.splice(0, list.length - MAX_PER_CELL);
  historyMap.set(k, list);
  persistHistory();
}

function snapshotAll(source = 'baseline') {
  for (const c of allCells()) {
    const txt = readCellText(c);
    if (typeof txt === 'string') recordSnapshot(c.id, txt, source);
  }
}

/* Pure line-level diff for inline AI Edit Review */
function renderLineDiff(oldText, newText) {
  const a = String(oldText || '').split('\n');
  const b = String(newText || '').split('\n');
  const ops = [];
  if (a.length * b.length <= 160000) {
    const m = a.length, n = b.length;
    const dp = Array.from({length: m + 1}, () => new Uint32Array(n + 1));
    for (let i = m - 1; i >= 0; i--) {
      for (let j = n - 1; j >= 0; j--) {
        dp[i][j] = a[i] === b[j] ? dp[i + 1][j + 1] + 1 : Math.max(dp[i + 1][j], dp[i][j + 1]);
      }
    }
    let i = 0, j = 0;
    while (i < m && j < n) {
      if (a[i] === b[j]) { ops.push(['eq', a[i++]]); j++; }
      else if (dp[i + 1][j] >= dp[i][j + 1]) ops.push(['del', a[i++]]);
      else ops.push(['add', b[j++]]);
    }
    while (i < m) ops.push(['del', a[i++]]);
    while (j < n) ops.push(['add', b[j++]]);
  } else {
    let p = 0;
    while (p < a.length && p < b.length && a[p] === b[p]) ops.push(['eq', a[p++]]);
    let ea = a.length - 1, eb = b.length - 1;
    const tail = [];
    while (ea >= p && eb >= p && a[ea] === b[eb]) { tail.push(['eq', a[ea--]]); eb--; }
    for (let i = p; i <= ea; i++) ops.push(['del', a[i]]);
    for (let j = p; j <= eb; j++) ops.push(['add', b[j]]);
    ops.push(...tail.reverse());
  }
  const prefix = {eq: '  ', del: '- ', add: '+ '};
  return el('pre', {cls: 'craft-diff-pre'}, ...ops.map(([k, line]) => el('div', {cls: `craft-diff-${k}`}, prefix[k] + line)));
}

async function restoreText(id, text) {
  const k = normId(id);
  try {
    await htmxPost('/set_contents_', {id_: '_' + k, ids: k, contents: JSON.stringify({[k]: text})});
    recordSnapshot(k, text, 'restore');
    const c = getCellEl(k);
    if (c) c.removeAttribute('data-diff');
    notify('Cell content restored.', 'success');
    return true;
  } catch (err) {
    notifyErr('Could not restore cell snapshot.', 'craftHistory');
    return false;
  }
}


### HTMX Capture & Inline AI Edit Review

In [ ]:
#| export_js_iife CRAFT.js craftHistory
#| eval: false
/* Hook /set_contents_ and /add_runq_ to capture snapshots */
on(document, 'htmx:configRequest', e => {
  const path = String(e.detail?.path || '');
  if (path.includes('/set_contents_')) {
    const raw = e.detail?.parameters?.contents;
    if (raw) {
      try {
        const map = JSON.parse(raw);
        for (const [id, txt] of Object.entries(map)) recordSnapshot(id, String(txt ?? ''), 'save');
      } catch {}
    }
  } else if (path.includes('/add_runq_')) {
    snapshotAll('pre-run');
  }
}, sig);

/* Inline AI Edit Review bars for data-diff="changed" | "deleted" */
const diffBaselines = new Map();

function syncDiffBars() {
  for (const c of allCells()) {
    const id = normId(c.id);
    const diff = c.getAttribute('data-diff');
    const isReview = diff === 'changed' || diff === 'deleted';
    const existing = c.querySelector(':scope > .craft-diff-bar');
    if (!isReview) {
      existing?.remove();
      diffBaselines.delete(id);
      continue;
    }
    if (existing) continue;
    const snaps = historyMap.get(id) || [];
    if (!diffBaselines.has(id)) {
      const preAi = [...snaps].reverse().find(s => s.source !== 'current' && s.source !== 'accept-ai') || snaps.at(-1);
      if (preAi) diffBaselines.set(id, preAi.text);
    }
    const hasBaseline = diffBaselines.has(id);
    const prevText = hasBaseline ? diffBaselines.get(id) : '';
    const diffBox = el('div', {cls: 'craft-diff-box', hidden: true});
    const bar = el('div', {cls: 'craft-diff-bar'},
      el('span', {cls: 'craft-diff-tag'}, diff === 'deleted' ? 'AI deleted cell' : 'AI modified cell'),
      el('button', {
        type: 'button', cls: 'uk-btn uk-btn-default uk-btn-xs',
        onClick: () => {
          diffBox.hidden = !diffBox.hidden;
          if (!diffBox.hidden) {
            const curRaw = readCellText(c);
            diffBox.replaceChildren(
              typeof curRaw === 'string'
                ? renderLineDiff(prevText, curRaw)
                : el('div', {cls: 'craft-empty-state'}, 'Open this cell to diff raw source.')
            );
          }
        },
      }, 'Toggle diff'),
      el('button', {
        type: 'button', cls: 'uk-btn uk-btn-primary uk-btn-xs',
        onClick: () => {
          const curRaw = readCellText(c);
          if (typeof curRaw === 'string') recordSnapshot(id, curRaw, 'accept-ai');
          diffBaselines.delete(id);
          c.removeAttribute('data-diff');
          bar.remove();
        },
      }, 'Accept'),
      hasBaseline ? el('button', {
        type: 'button', cls: 'uk-btn uk-btn-default uk-btn-xs craft-btn-danger',
        onClick: async () => { if (await restoreText(id, prevText)) diffBaselines.delete(id); },
      }, 'Revert') : null,
      diffBox,
    );
    c.prepend(bar);
  }
}

### Cell History Modal & Lifecycle

In [ ]:
#| export_js_iife CRAFT.js craftHistory
#| eval: false
/* Cell History Modal, Menu, Hotkey & Lifecycle */
let modal = null;

function openHistory(cell) {
  const target = cell || primaryCell();
  if (!target) return notifyErr('Select a cell first to view its history.', 'craftHistory');
  const id = normId(target.id);
  const curTxt = readCellText(target);
  if (typeof curTxt === 'string') recordSnapshot(id, curTxt, 'current');
  const snaps = [...(historyMap.get(id) || [])].reverse();
  if (!modal) {
    modal = mountModal('hist_modal', {signal: ac.signal});
    on(modal.done, 'click', modal.close, sig);
  }
  modal.cellLabel.textContent = `#${id.slice(0, 8)} · ${snaps.length} snapshot(s)`;
  modal.list.replaceChildren(...(snaps.length ? snaps.map((s, i) => {
    const tStr = new Date(s.ts).toLocaleTimeString([], {hour: '2-digit', minute: '2-digit', second: '2-digit'});
    return el('div', {cls: 'craft-hist-entry'},
      el('div', {cls: 'craft-hist-meta'},
        el('span', {cls: 'craft-hist-time'}, `${tStr} · ${s.source}`),
        i > 0 ? el('button', {
          type: 'button', cls: 'uk-btn uk-btn-default uk-btn-xs',
          onClick: async () => { if (await restoreText(id, s.text)) modal.close(); },
        }, 'Restore') : el('span', {cls: 'craft-hist-cur'}, 'latest'),
      ),
      el('pre', {cls: 'craft-hist-snip'}, s.text.slice(0, 400) || '(empty)'),
    );
  }) : [el('div', {cls: 'craft-empty-state'}, 'No snapshots recorded yet in this session.')]));
  modal.open();
}

const offMenu = window.craftActionMenus?.registerProvider?.(ctx => {
  if (ctx.isOutput || !ctx.cell) return [];
  return [{label: 'Cell history…', onClick: () => openHistory(ctx.cell)}];
});

window.__hotkeyRegister?.({id: 'craft-cell-history', label: 'Cell version history (session)', group: 'CRAFT · History', combo: '', handler: () => { openHistory(); return true; }});

const offDom = onDomChange(() => syncDiffBars(), {attrs: ['data-diff'], delay: 120, inDialog: true});
snapshotAll('initial');
syncDiffBars();

window.craftHistory = {
  open: openHistory,
  snapshots: id => [...(historyMap.get(normId(id)) || [])],
  recordSnapshot,
  restoreText,
  disable() {
    ac.abort(); offDom?.(); offMenu?.();
    modal?.destroy();
    qsa('.craft-diff-bar').forEach(b => b.remove());
    delete window.craftHistory;
  },
};

## Cell History & AI Diff CSS (`#| export_css`)

In [ ]:
#| export_css CRAFT.css
#| eval: false
/* ── Cell Version History & AI Edit Review (17_cell_history) ──────── */
.craft-diff-bar {
  display: flex; flex-wrap: wrap; align-items: center; gap: 6px;
  padding: 5px 8px; margin-bottom: 4px; border-radius: 5px;
  background: var(--craft-recessed); border: 1px solid var(--craft-amber); font-size: 11px;
}
.craft-diff-tag { font-weight: 600; color: var(--craft-amber); margin-right: auto; }
.craft-diff-box { width: 100%; margin-top: 6px; }
.craft-diff-pre {
  margin: 0; padding: 6px 8px; max-height: 240px; overflow: auto;
  background: var(--craft-bg); border: 1px solid var(--craft-border); border-radius: 4px;
  font-family: var(--craft-code-family); font-size: 10.5px; line-height: 1.35;
}
.craft-diff-add { color: var(--craft-green); background: color-mix(in srgb, var(--craft-green) 12%, transparent); }
.craft-diff-del { color: var(--craft-red); background: color-mix(in srgb, var(--craft-red) 12%, transparent); }
.craft-diff-eq { color: var(--craft-muted); }

.craft-hist-card { width: min(640px, 92vw); max-height: 78vh; display: flex; flex-direction: column; overflow: hidden; }
.craft-hist-list { overflow-y: auto; padding: 10px 14px; display: flex; flex-direction: column; gap: 8px; }
.craft-hist-entry { border: 1px solid var(--craft-border); border-radius: 6px; background: var(--craft-bg); overflow: hidden; }
.craft-hist-meta {
  display: flex; align-items: center; justify-content: space-between;
  padding: 5px 8px; background: var(--craft-recessed); border-bottom: 1px solid var(--craft-border-soft); font-size: 11px;
}
.craft-hist-time { font-family: var(--craft-code-family); color: var(--craft-muted); }
.craft-hist-cur { font-size: 10px; color: var(--craft-green); text-transform: uppercase; font-weight: 600; }
.craft-hist-snip {
  margin: 0; padding: 6px 8px; max-height: 120px; overflow: auto; white-space: pre-wrap;
  font-family: var(--craft-code-family); font-size: 10.5px; color: var(--craft-fg);
}

Verify `17_cell_history.ipynb` exports `window.craftHistory`, its template, and CSS.

In [ ]:
import json, os, subprocess
from craft_dev.auth import REPO_ROOT
from craft_dev.build import extract_nb_assets

assets = extract_nb_assets(REPO_ROOT/'nbs/craft/17_cell_history.ipynb')
js = '\n'.join(code for _, code in assets['CRAFT.js'])
css = '\n'.join(code for _, code in assets['CRAFT.css'])
for sym in ('window.craftHistory', 'hist_modal', 'renderLineDiff', 'syncDiffBars', 'mountModal', 'craft-empty-state', 'craft-btn-danger'):
    assert sym in js, f"Missing {sym}"
assert '.craft-diff-bar' in css and '.craft-hist-card' in css

node_script = r"""
const src = process.env.HIST_JS;
const i0 = src.indexOf('function renderLineDiff');
const i1 = src.indexOf('async function restoreText');
const el = (tag, attrs = {}, ...children) => ({tag, cls: attrs.cls || '', children});
const renderLineDiff = new Function('el', src.slice(i0, i1) + '; return renderLineDiff;')(el);

const opsOf = (a, b) => renderLineDiff(a, b).children.map(c => [c.cls, c.children[0]]);

// 1. Single insertion in the middle must keep surrounding lines equal (real LCS)
const ins = opsOf('a\nb\nc\nd', 'a\nb\nNEW\nc\nd');
if (JSON.stringify(ins) !== JSON.stringify([
  ['craft-diff-eq', '  a'], ['craft-diff-eq', '  b'],
  ['craft-diff-add', '+ NEW'], ['craft-diff-eq', '  c'], ['craft-diff-eq', '  d']
])) throw new Error('LCS insertion mismatch: ' + JSON.stringify(ins));

// 2. Deletion + replacement
const rep = opsOf('x\ny\nz', 'x\nY');
const kinds = rep.map(([k]) => k);
if (kinds[0] !== 'craft-diff-eq' || !kinds.includes('craft-diff-del') || !kinds.includes('craft-diff-add'))
  throw new Error('Replacement mismatch: ' + JSON.stringify(rep));

// 3. Large-input fallback (>160k cell pairs) preserves prefix & suffix
const base = Array.from({length: 410}, (_, i) => 'line-' + i);
const mod = base.slice();
mod[200] = 'CHANGED-200';
const big = opsOf(base.join('\n'), mod.join('\n'));
const dels = big.filter(([k]) => k === 'craft-diff-del');
const adds = big.filter(([k]) => k === 'craft-diff-add');
if (dels.length !== 1 || adds.length !== 1 || dels[0][1] !== '- line-200' || adds[0][1] !== '+ CHANGED-200')
  throw new Error('Large fallback mismatch');

process.stdout.write(JSON.stringify({ok: true, bigLen: big.length}));
"""
res = subprocess.run(['node', '-e', node_script], env={**os.environ, 'HIST_JS': js}, capture_output=True, text=True, check=True)
assert json.loads(res.stdout)['ok']
